In [29]:
import os
from towbintools.foundation.file_handling import read_filemap, add_dir_to_experiment_filemap
import polars as pl

experiment_path = "/mnt/towbin.data/shared/spsalmon/20240628_153634_380_LIPSI_40x_397_405_new_round_chambers"
report_folder = os.path.join(experiment_path, "analysis", "report")
segmentation_folder = os.path.join(experiment_path, "analysis", "ch1_seg_stardist_shifted")
pads = ['pad1', 'pad2']

In [30]:
# add the nuclear segmentation to the filemaps
for pad in pads:
    report_folder_pad = os.path.join(report_folder, pad)
    experiment_filemap = read_filemap(os.path.join(report_folder_pad, "analysis_filemap_annotated.csv"))
    # backup the original filemap
    experiment_filemap.write_csv(os.path.join(report_folder_pad, "analysis_filemap_annotated_backup.csv"))
    segmentation_folder_pad = os.path.join(segmentation_folder, pad)
    experiment_filemap = add_dir_to_experiment_filemap(experiment_filemap, segmentation_folder_pad, "ch1_seg_stardist_shifted")
    experiment_filemap.write_csv(os.path.join(report_folder_pad, "analysis_filemap_annotated.csv"))
    print(experiment_filemap.head())


shape: (5, 36)
┌──────┬───────┬──────────────┬──────────────┬───┬───────────┬────────┬──────────────┬─────────────┐
│ Time ┆ Point ┆ raw          ┆ volume_ch3_d ┆ … ┆ OutOfFood ┆ Ignore ┆ ch1_seg_star ┆ ch1_seg_sta │
│ ---  ┆ ---   ┆ ---          ┆ ouble_thresh ┆   ┆ ---       ┆ ---    ┆ dist         ┆ rdist_shift │
│ i64  ┆ i64   ┆ str          ┆ old          ┆   ┆ str       ┆ bool   ┆ ---          ┆ ed          │
│      ┆       ┆              ┆ ---          ┆   ┆           ┆        ┆ str          ┆ ---         │
│      ┆       ┆              ┆ str          ┆   ┆           ┆        ┆              ┆ str         │
╞══════╪═══════╪══════════════╪══════════════╪═══╪═══════════╪════════╪══════════════╪═════════════╡
│ 0    ┆ 0     ┆ /mnt/towbin. ┆              ┆ … ┆           ┆ false  ┆ /mnt/towbin. ┆ /mnt/towbin │
│      ┆       ┆ data/shared/ ┆              ┆   ┆           ┆        ┆ data/shared/ ┆ .data/share │
│      ┆       ┆ spsalm…      ┆              ┆   ┆           ┆        ┆ spsa

In [31]:
previous_max_point = 0
filemaps = []
for pad in pads:
    report_folder_pad = os.path.join(report_folder, pad)
    experiment_filemap = read_filemap(os.path.join(report_folder_pad, "analysis_filemap_annotated.csv"))
    experiment_filemap = experiment_filemap.with_columns(pl.lit(pad).alias("Pad"))
    experiment_filemap = experiment_filemap.with_columns((pl.col("Point") + previous_max_point).alias("Point"))
    previous_max_point = experiment_filemap.select(pl.col("Point").max()).item() + 1

    print(experiment_filemap.head())
    print(experiment_filemap.columns)
    filemaps.append(experiment_filemap)

all_filemap = pl.concat(filemaps, how="diagonal_relaxed")
print(all_filemap.head())

all_filemap.write_csv(os.path.join(report_folder, "analysis_filemap_annotated.csv"))

shape: (5, 37)
┌──────┬───────┬───────────────┬───────────────┬───┬────────┬───────────────┬───────────────┬──────┐
│ Time ┆ Point ┆ raw           ┆ volume_ch3_do ┆ … ┆ Ignore ┆ ch1_seg_stard ┆ ch1_seg_stard ┆ Pad  │
│ ---  ┆ ---   ┆ ---           ┆ uble_threshol ┆   ┆ ---    ┆ ist           ┆ ist_shifted   ┆ ---  │
│ i64  ┆ i64   ┆ str           ┆ d             ┆   ┆ bool   ┆ ---           ┆ ---           ┆ str  │
│      ┆       ┆               ┆ ---           ┆   ┆        ┆ str           ┆ str           ┆      │
│      ┆       ┆               ┆ str           ┆   ┆        ┆               ┆               ┆      │
╞══════╪═══════╪═══════════════╪═══════════════╪═══╪════════╪═══════════════╪═══════════════╪══════╡
│ 0    ┆ 0     ┆ /mnt/towbin.d ┆ null          ┆ … ┆ false  ┆ /mnt/towbin.d ┆ /mnt/towbin.d ┆ pad1 │
│      ┆       ┆ ata/shared/sp ┆               ┆   ┆        ┆ ata/shared/sp ┆ ata/shared/sp ┆      │
│      ┆       ┆ salm…         ┆               ┆   ┆        ┆ salm…         